# Fine-tuning Qwen3-8B para Visual FoxPro 9

QLoRA em 4 bits sobre `train_final.jsonl` (5.409 exemplos) com validação em `val_final.jsonl` (285 exemplos).

O corpus é heterogêneo de propósito e treina duas competências numa passada só:

| Corpus | Linhas | Prompt de sistema | O que ensina |
|---|---|---|---|
| Conhecimento | 4.609 | assistente | fichas de assinatura e exemplos de código VFP9 |
| Agêntico | 800 | agente | uso das ferramentas `list_files`, `search_text`, `read_file`, `edit_file` |

**Defesas contra overfitting** (o problema da rodada anterior), todas nas células 11 e 12:
avaliação a cada 50 passos, parada antecipada por `eval_loss`, restauração automática do melhor
checkpoint, LoRA de posto baixo, `weight_decay`, NEFTune e teto de 3 épocas que provavelmente
não será atingido.

**Antes de rodar:** selecione um runtime com GPU em *Ambiente de execução → Alterar o tipo de
ambiente de execução*. L4 ou A100 são confortáveis; a T4 funciona, porém sem bf16 e bem mais lenta.

## 1. Google Drive, cache e diretórios

O cache da Hugging Face fica no Drive para não baixar de novo os ~16 GB do modelo a cada sessão. Em troca,
o carregamento dos pesos é mais lento do que no disco local, que é efêmero.

In [ ]:
import os

from google.colab import drive

# ============================================================
# GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")

# Projeto EXCLUSIVO do Qwen3-8B
PROJECT_DIR = "/content/drive/MyDrive/VFP-LLM-Qwen3-8B"

os.makedirs(
    PROJECT_DIR,
    exist_ok=True,
)

# ============================================================
# CACHE DA HUGGING FACE
# ============================================================

HF_HOME = os.path.join(
    PROJECT_DIR,
    "huggingface",
)

HF_HUB_CACHE = os.path.join(
    HF_HOME,
    "hub",
)

HF_XET_CACHE = os.path.join(
    HF_HOME,
    "xet",
)

HF_DATASETS_CACHE = os.path.join(
    HF_HOME,
    "datasets",
)

os.environ["HF_HOME"] = HF_HOME
os.environ["HF_HUB_CACHE"] = HF_HUB_CACHE
os.environ["HF_XET_CACHE"] = HF_XET_CACHE
os.environ["HF_DATASETS_CACHE"] = HF_DATASETS_CACHE

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ============================================================
# DIRETÓRIOS
# ============================================================

TRAINING_DIR = os.path.join(
    PROJECT_DIR,
    "training",
)

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR,
    "checkpoints",
)

FINAL_MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "final_model",
)

MERGED_MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "merged_model",
)

LOG_DIR = os.path.join(
    PROJECT_DIR,
    "logs",
)

for directory in [
    TRAINING_DIR,
    CHECKPOINT_DIR,
    FINAL_MODEL_DIR,
    LOG_DIR,
    HF_HOME,
    HF_HUB_CACHE,
    HF_XET_CACHE,
    HF_DATASETS_CACHE,
]:
    os.makedirs(
        directory,
        exist_ok=True,
    )

os.chdir(PROJECT_DIR)

print("Projeto Qwen3-8B:", PROJECT_DIR)

## 2. Dependências

Se o Colab pedir para reiniciar a sessão depois desta célula, reinicie e execute de novo a partir
da célula 1. As variáveis de ambiente precisam existir antes de qualquer import da Hugging Face.

In [ ]:
%pip install -q -U \
    "transformers>=4.51" \
    "accelerate>=1.0" \
    "peft>=0.14" \
    "bitsandbytes>=0.45" \
    "datasets>=3.0" \
    "matplotlib"

## 3. Imports e detecção de hardware

A escolha entre bf16 e fp16 vem da GPU sorteada. Ampere ou superior (L4, A100) usa bf16, que é mais
estável no treino; a T4 é Turing e só tem fp16.

In [ ]:
import json
import math
import time

import torch
import transformers
import peft

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

# ============================================================
# REPRODUTIBILIDADE
# ============================================================

SEED = 42

set_seed(SEED)

# ============================================================
# HARDWARE
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "Nenhuma GPU visível. Ambiente de execução -> Alterar o tipo de ambiente de execução -> GPU."
    )

NOME_GPU = torch.cuda.get_device_name(0)
MEMORIA_GPU = torch.cuda.get_device_properties(0).total_memory / 1024**3
USA_BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USA_BF16 else torch.float16

print("transformers:", transformers.__version__)
print("peft........:", peft.__version__)
print("torch.......:", torch.__version__)
print()
print(f"GPU.........: {NOME_GPU} ({MEMORIA_GPU:.1f} GB)")
print(f"precisão....: {'bf16' if USA_BF16 else 'fp16'}")

## 4. Localizar os arquivos de treino

A célula procura `train_final.jsonl` e `val_final.jsonl` no Drive e em `/content`, que é onde o
painel *Arquivos* do Colab deposita os envios. Achando fora do Drive, copia para lá — o `/content`
é apagado quando a sessão cai, e a cópia evita reenviar tudo depois. Se não achar em lugar nenhum,
abre o seletor de upload.

In [ ]:
import shutil

# ============================================================
# ARQUIVOS DE TREINO
# ============================================================

NOMES_JSONL = (
    "train_final.jsonl",
    "val_final.jsonl",
)

# O Drive vem primeiro para reaproveitar o que já foi copiado numa sessão anterior;
# /content é onde o painel "Arquivos" deposita os envios e some ao desconectar.
ORIGENS = [
    TRAINING_DIR,
    "/content",
    os.getcwd(),
    "/content/drive/MyDrive",
]


def localizar(nome: str) -> str | None:
    """Primeiro caminho existente para o arquivo, ou None se não estiver em nenhuma origem."""
    for origem in ORIGENS:
        caminho = os.path.join(origem, nome)

        if os.path.isfile(caminho):
            return caminho

    return None


encontrados = {nome: localizar(nome) for nome in NOMES_JSONL}
faltando = [nome for nome, caminho in encontrados.items() if caminho is None]

if faltando:
    from google.colab import files

    print("Não encontrei:", ", ".join(faltando))
    print("Selecione os arquivos abaixo.\n")

    enviados = files.upload()

    for nome in faltando:
        if nome not in enviados:
            raise FileNotFoundError(
                f"{nome} não foi enviado. Execute a célula de novo e selecione os dois arquivos."
            )

        destino = os.path.join(TRAINING_DIR, nome)

        with open(destino, "wb") as arquivo:
            arquivo.write(enviados[nome])

        encontrados[nome] = destino

# Consolida no Drive para sobreviver à próxima desconexão do Colab.
for nome, origem in encontrados.items():
    destino = os.path.join(TRAINING_DIR, nome)

    if os.path.abspath(origem) != os.path.abspath(destino):
        print(f"copiando {origem} -> {destino}")
        shutil.copyfile(origem, destino)

TRAIN_FILE = os.path.join(
    TRAINING_DIR,
    "train_final.jsonl",
)

VAL_FILE = os.path.join(
    TRAINING_DIR,
    "val_final.jsonl",
)

print()

for caminho in (TRAIN_FILE, VAL_FILE):
    with open(caminho, encoding="utf-8") as arquivo:
        linhas = sum(1 for _ in arquivo)

    print(
        f"{os.path.basename(caminho):20s} "
        f"{os.path.getsize(caminho) / 1024**2:6.2f} MB  {linhas:5d} linhas"
    )

## 5. Carga e normalização

O JSONL não é lido com `load_dataset` porque o esquema varia entre as linhas: só as agênticas têm a
chave `tools`, e nelas o `content` do assistente é `null`. O Arrow tentaria unificar isso e falharia.

Duas normalizações são obrigatórias para o template de chat do Qwen:

- `content: null` vira string vazia, senão o Jinja concatena o literal `None` no texto de treino;
- `arguments` deixa de ser string JSON e vira dicionário, porque templates antigos aplicam `tojson`
  sem verificar o tipo e produziriam uma string escapada dentro do `<tool_call>`.

In [ ]:
def carregar(caminho: str) -> list[dict]:
    """Lê o JSONL e devolve os registros já normalizados para o template do Qwen."""
    registros = []

    with open(caminho, encoding="utf-8") as arquivo:
        for numero, linha in enumerate(arquivo, start=1):
            linha = linha.strip()

            if not linha:
                continue

            registro = json.loads(linha)
            mensagens = registro["messages"]

            for mensagem in mensagens:
                if mensagem.get("content") is None:
                    mensagem["content"] = ""

                for chamada in mensagem.get("tool_calls") or []:
                    argumentos = chamada["function"]["arguments"]

                    if isinstance(argumentos, str):
                        chamada["function"]["arguments"] = json.loads(argumentos)

            if mensagens[-1]["role"] != "assistant":
                raise ValueError(f"{caminho}:{numero} não termina em assistant")

            registro["corpus"] = "agentico" if "tools" in registro else "conhecimento"
            registros.append(registro)

    return registros


treino_bruto = carregar(TRAIN_FILE)
validacao_bruta = carregar(VAL_FILE)

for nome, registros in (("treino", treino_bruto), ("validação", validacao_bruta)):
    agenticos = sum(1 for r in registros if r["corpus"] == "agentico")
    print(f"{nome:10s} {len(registros):5d} exemplos  ({agenticos} agênticos, {len(registros) - agenticos} de conhecimento)")

## 6. Tokenizer e máscara de perda

O diálogo é renderizado uma única vez pelo template oficial do Qwen3 e depois recortado nos
delimitadores `<|im_start|>assistant` e `<|im_end|>`. Só os trechos produzidos pelo assistente
entram na perda; sistema, usuário e retorno de ferramenta recebem `-100`.

Mascarar o papel `tool` é o ponto mais importante: aquele texto é saída do ambiente, e treinar
nele ensinaria o modelo a inventar o conteúdo dos arquivos em vez de ir buscá-lo.

Duas sutilezas do template do Qwen3 que este recorte resolve:

- **Não dá para renderizar prefixos sucessivos e comparar.** O template injeta um bloco de
  raciocínio vazio no *último* turno do assistente, então a renderização de um prefixo não é
  prefixo da renderização seguinte. O recorte trabalha sobre um render único e escapa disso.
- **Esse bloco vazio é removido.** Como o corpus não tem raciocínio, mantê-lo ensinaria o modelo
  a reemitir `<think></think>` — que na inferência o próprio prompt já traria, duplicando-o.
  Removido, cada turno do assistente fica na forma `<|im_start|>assistant\n{conteúdo}`, idêntica
  ao prompt de geração padrão. **Consequência: não passe `enable_thinking=False` ao servir.**

O `<|im_end|>` fica dentro do alvo de propósito, para o modelo aprender onde parar.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

IGNORAR = -100

ABRE_ASSISTENTE = "<|im_start|>assistant\n"
BLOCO_VAZIO = "<think>\n\n</think>\n\n"
FIM_TURNO = "<|im_end|>"


def renderizar(registro: dict) -> str:
    """Aplica o template oficial e remove o bloco de raciocínio vazio injetado no último turno."""
    texto = tokenizer.apply_chat_template(
        registro["messages"],
        tools=registro.get("tools"),
        tokenize=False,
        add_generation_prompt=False,
    )

    return texto.replace(ABRE_ASSISTENTE + BLOCO_VAZIO, ABRE_ASSISTENTE)


def repartir(texto: str) -> list[tuple[str, bool]]:
    """Quebra o texto em trechos alternados de contexto e de resposta do assistente."""
    pedacos: list[tuple[str, bool]] = []
    posicao = 0

    while True:
        abertura = texto.find(ABRE_ASSISTENTE, posicao)

        if abertura < 0:
            pedacos.append((texto[posicao:], False))
            break

        inicio = abertura + len(ABRE_ASSISTENTE)
        fim = texto.find(FIM_TURNO, inicio)

        if fim < 0:
            raise ValueError("turno do assistente sem <|im_end|>")

        pedacos.append((texto[posicao:inicio], False))
        pedacos.append((texto[inicio : fim + len(FIM_TURNO)], True))

        posicao = fim + len(FIM_TURNO)

    return [(trecho, treinar) for trecho, treinar in pedacos if trecho]


def montar_exemplo(registro: dict) -> dict:
    """Tokeniza o diálogo e devolve rótulos ativos apenas nos turnos do assistente."""
    texto = renderizar(registro)
    pedacos = repartir(texto)

    if "".join(trecho for trecho, _ in pedacos) != texto:
        raise RuntimeError("o recorte não reconstrói o texto renderizado")

    ids: list[int] = []
    rotulos: list[int] = []

    for trecho, treinar in pedacos:
        parte = tokenizer(trecho, add_special_tokens=False)["input_ids"]

        ids.extend(parte)
        rotulos.extend(parte if treinar else [IGNORAR] * len(parte))

    if all(rotulo == IGNORAR for rotulo in rotulos):
        raise RuntimeError("exemplo sem nenhum token supervisionado")

    return {"input_ids": ids, "labels": rotulos}


amostra = montar_exemplo(treino_bruto[0])

print("tokens do primeiro exemplo:", len(amostra["input_ids"]))
print("tokens que entram na perda:", sum(1 for r in amostra["labels"] if r != IGNORAR))

## 7. Tokenização completa e comprimento de sequência

O `MAX_SEQ_LEN` é derivado dos dados: o menor múltiplo de 256 que acomoda o exemplo mais longo.
Cortar pelo comprimento médio seria um erro, porque os traces agênticos são quatro vezes mais longos
que as fichas e o corte cairia justamente no turno final — o modelo aprenderia a chamar ferramentas
e nunca concluir.

In [ ]:
TETO_SEQ_LEN = 4096

inicio = time.time()

treino_tokens = [montar_exemplo(r) for r in treino_bruto]
validacao_tokens = [montar_exemplo(r) for r in validacao_bruta]

print(f"tokenização concluída em {time.time() - inicio:.1f}s\n")

comprimentos = [len(e["input_ids"]) for e in treino_tokens + validacao_tokens]
maior = max(comprimentos)

MAX_SEQ_LEN = min(TETO_SEQ_LEN, math.ceil(maior / 256) * 256)


def percentil(valores: list[int], posicao: int) -> int:
    ordenados = sorted(valores)
    return ordenados[min(len(ordenados) - 1, int(len(ordenados) * posicao / 100))]


for nome, exemplos in (("treino", treino_tokens), ("validação", validacao_tokens)):
    for corpus in ("agentico", "conhecimento"):
        brutos = treino_bruto if nome == "treino" else validacao_bruta
        tamanhos = [
            len(e["input_ids"]) for e, r in zip(exemplos, brutos) if r["corpus"] == corpus
        ]
        print(
            f"{nome:10s} {corpus:13s} n={len(tamanhos):5d}  "
            f"mediana={percentil(tamanhos, 50):5d}  p95={percentil(tamanhos, 95):5d}  máx={max(tamanhos):5d}"
        )

print(f"\nMAX_SEQ_LEN = {MAX_SEQ_LEN} (maior exemplo: {maior} tokens)")

acima_do_teto = sum(1 for c in comprimentos if c > MAX_SEQ_LEN)

if acima_do_teto:
    print(
        f"\nAVISO: {acima_do_teto} exemplos passam de {MAX_SEQ_LEN} tokens e serão descartados, "
        "não truncados. Um trace cortado no meio ensinaria o modelo a nunca concluir."
    )

treino_tokens = [e for e in treino_tokens if len(e["input_ids"]) <= MAX_SEQ_LEN]
validacao_tokens = [e for e in validacao_tokens if len(e["input_ids"]) <= MAX_SEQ_LEN]

dataset_treino = Dataset.from_list(treino_tokens)
dataset_validacao = Dataset.from_list(validacao_tokens)

supervisionados = sum(
    1 for e in treino_tokens for r in e["labels"] if r != IGNORAR
)

print(f"\ntokens no treino.......: {sum(len(e['input_ids']) for e in treino_tokens):,}")
print(f"tokens supervisionados.: {supervisionados:,}")

## 8. Conferência visual da máscara

Vale gastar trinta segundos aqui. A célula mostra, num trace agêntico, exatamente o que o modelo
aprende a produzir. Devem aparecer as chamadas `<tool_call>` e a resposta final — e nada do conteúdo
devolvido pelas ferramentas.

In [ ]:
indice_agentico = next(i for i, r in enumerate(treino_bruto) if r["corpus"] == "agentico")
exemplo = montar_exemplo(treino_bruto[indice_agentico])

treinados = [t for t, r in zip(exemplo["input_ids"], exemplo["labels"]) if r != IGNORAR]
mascarados = [t for t, r in zip(exemplo["input_ids"], exemplo["labels"]) if r == IGNORAR]

print("=" * 70)
print("TREINADO (o modelo aprende a gerar isto)")
print("=" * 70)
print(tokenizer.decode(treinados))

print()
print("=" * 70)
print("MASCARADO (contexto; não entra na perda)")
print("=" * 70)
print(tokenizer.decode(mascarados)[:1200], "...")

print()
print(f"proporção treinada: {len(treinados) / len(exemplo['input_ids']):.1%}")

## 9. Modelo base em 4 bits

NF4 com dupla quantização. Os ~16 GB do Qwen3-8B em bf16 caem para cerca de 5,5 GB de pesos, o que
deixa folga para ativações mesmo numa T4 de 16 GB.

In [ ]:
quantizacao = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)

# A partir do transformers 5 o parâmetro torch_dtype passou a se chamar dtype.
chave_dtype = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"

modelo = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantizacao,
    attn_implementation="sdpa",
    device_map={"": 0},
    trust_remote_code=True,
    **{chave_dtype: DTYPE},
)

modelo.config.use_cache = False

modelo = prepare_model_for_kbit_training(
    modelo,
    use_gradient_checkpointing=True,
)

print(f"memória ocupada: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 10. Adaptadores LoRA

Posto 16 em todas as projeções lineares. Com 5.409 exemplos, um posto maior daria capacidade de
memorizar o conjunto de treino em vez de generalizar — é a primeira alavanca contra o overfitting da
rodada anterior. Se depois de treinar a perda de validação ainda estiver caindo junto com a de
treino no fim, aí sim vale subir para 32.

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

modelo = get_peft_model(
    modelo,
    lora,
)

modelo.print_trainable_parameters()

## 11. Configuração do treino

As decisões que atacam o overfitting:

- **Parada antecipada.** Avaliação a cada 50 passos e interrupção após 3 avaliações sem melhora na
  perda de validação. As 3 épocas são um teto que provavelmente não será alcançado.
- **Melhor checkpoint restaurado.** `load_best_model_at_end` devolve os pesos do passo com a menor
  perda de validação, não os do último passo. Mesmo que o treino siga até o fim, o que você salva é
  o melhor ponto.
- **Taxa de aprendizado contida.** `1e-4` com decaimento cosseno e 3% de aquecimento.
- **`weight_decay` e NEFTune.** O ruído do NEFTune nos embeddings é uma regularização barata que
  costuma ajudar justamente em conjuntos pequenos de instrução.
- **Lotes misturados.** Sem agrupamento por comprimento: juntar os traces agênticos nos mesmos
  lotes atrapalharia: é misturar os dois corpora em cada passo que ensina o modelo a alternar de
  comportamento conforme o prompt de sistema.

O `TrainingArguments` encolheu bastante na série 5 do transformers, e o Colab atualiza a biblioteca
sem aviso. Por isso os parâmetros são montados como dicionário e filtrados pelos campos que a versão
instalada realmente expõe, com aviso do que foi descartado — em vez de estourar um `TypeError` a
cada remoção.

In [ ]:
import dataclasses

LOTE_POR_DISPOSITIVO = 2
ACUMULACAO = 8

lote_efetivo = LOTE_POR_DISPOSITIVO * ACUMULACAO
passos_por_epoca = math.ceil(len(dataset_treino) / lote_efetivo)

CAMPOS = {campo.name for campo in dataclasses.fields(TrainingArguments)}

parametros = {
    "output_dir": CHECKPOINT_DIR,
    # ---------- duração ----------
    "num_train_epochs": 3,
    "per_device_train_batch_size": LOTE_POR_DISPOSITIVO,
    "per_device_eval_batch_size": LOTE_POR_DISPOSITIVO,
    "gradient_accumulation_steps": ACUMULACAO,
    # ---------- otimização ----------
    "learning_rate": 1e-4,
    "lr_scheduler_type": "cosine",
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "optim": "paged_adamw_8bit",
    "neftune_noise_alpha": 5,
    # ---------- precisão e memória ----------
    "bf16": USA_BF16,
    "fp16": not USA_BF16,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "dataloader_num_workers": 2,
    # ---------- avaliação e checkpoints ----------
    "eval_strategy": "steps",
    "eval_steps": 50,
    "save_strategy": "steps",
    "save_steps": 50,
    "save_total_limit": 3,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "logging_steps": 10,
    "report_to": "none",
    "seed": SEED,
    "label_names": ["labels"],
}

# Aquecimento: até a 5.0 era warmup_ratio; depois os dois viraram warmup_steps, que
# passou a aceitar fração além de contagem absoluta.
if "warmup_ratio" in CAMPOS:
    parametros["warmup_ratio"] = 0.03
else:
    parametros["warmup_steps"] = 0.03

# Removido nas versões novas, onde não agrupar já é o comportamento único.
if "group_by_length" in CAMPOS:
    parametros["group_by_length"] = False

# logging_dir saiu do TrainingArguments e virou variável de ambiente do TensorBoard.
os.environ["TENSORBOARD_LOGGING_DIR"] = LOG_DIR

if "logging_dir" in CAMPOS:
    parametros["logging_dir"] = LOG_DIR

ignorados = sorted(set(parametros) - CAMPOS)

for nome in ignorados:
    del parametros[nome]

if ignorados:
    print("descartados por não existirem no transformers instalado:", ", ".join(ignorados))
    print()

argumentos = TrainingArguments(**parametros)

colecionador = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=IGNORAR,
    pad_to_multiple_of=8,
)

treinador = Trainer(
    model=modelo,
    args=argumentos,
    train_dataset=dataset_treino,
    eval_dataset=dataset_validacao,
    data_collator=colecionador,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"transformers.......: {transformers.__version__}")
print(f"lote efetivo.......: {lote_efetivo}")
print(f"passos por época...: {passos_por_epoca}")
print(f"teto de passos.....: {passos_por_epoca * 3}")
print(f"avaliações previstas: {passos_por_epoca * 3 // 50}")

## 12. Treinar

Retoma automaticamente do último checkpoint no Drive, então uma desconexão do Colab custa no máximo
50 passos. Basta reexecutar o notebook desde o início e chegar aqui de novo.

In [ ]:
ultimo_checkpoint = None

if os.path.isdir(CHECKPOINT_DIR):
    ultimo_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if ultimo_checkpoint:
    print("retomando de:", ultimo_checkpoint)

resultado = treinador.train(resume_from_checkpoint=ultimo_checkpoint)

print()
print("passos executados:", resultado.global_step)
print(f"perda final de treino: {resultado.training_loss:.4f}")

## 13. Curvas e diagnóstico

O sinal de overfitting é a perda de validação virar para cima enquanto a de treino continua caindo.
O ponto mínimo da curva laranja é o checkpoint que ficou carregado.

In [ ]:
import matplotlib.pyplot as plt

historico = treinador.state.log_history

treino = [(h["step"], h["loss"]) for h in historico if "loss" in h]
validacao = [(h["step"], h["eval_loss"]) for h in historico if "eval_loss" in h]

figura, eixo = plt.subplots(figsize=(11, 5))

eixo.plot(*zip(*treino), label="treino", alpha=0.6, linewidth=1)
eixo.plot(*zip(*validacao), label="validação", marker="o", linewidth=2)

melhor_passo, melhor_perda = min(validacao, key=lambda par: par[1])

eixo.axvline(melhor_passo, linestyle="--", color="green", alpha=0.7)
eixo.annotate(
    f"melhor: passo {melhor_passo}\neval_loss {melhor_perda:.4f}",
    xy=(melhor_passo, melhor_perda),
    xytext=(10, 20),
    textcoords="offset points",
    color="green",
)

eixo.set_xlabel("passo")
eixo.set_ylabel("perda")
eixo.set_title("Qwen3-8B QLoRA — VFP9")
eixo.legend()
eixo.grid(alpha=0.3)

plt.tight_layout()
plt.show()

ultima_perda = validacao[-1][1]

print(f"melhor eval_loss...: {melhor_perda:.4f} (passo {melhor_passo})")
print(f"última eval_loss...: {ultima_perda:.4f} (passo {validacao[-1][0]})")

if ultima_perda > melhor_perda * 1.02:
    print(
        "\nA validação piorou depois do mínimo: houve overfitting, e a parada antecipada fez o "
        "trabalho dela. Os pesos carregados são os do melhor passo."
    )
elif melhor_passo == validacao[-1][0]:
    print(
        "\nA validação ainda estava caindo no fim. Sobrou capacidade: vale aumentar as épocas ou "
        "o posto do LoRA para 32."
    )
else:
    print("\nCurva saudável, com mínimo bem definido.")

## 14. Salvar o adaptador no Drive

O que é salvo é o adaptador LoRA, de algumas centenas de megabytes, e não o modelo inteiro. Para
servir, basta carregar o Qwen3-8B base e aplicar este diretório por cima — vLLM e PEFT fazem isso
nativamente.

In [ ]:
treinador.model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

resumo = {
    "modelo_base": MODEL_ID,
    "max_seq_len": MAX_SEQ_LEN,
    "exemplos_treino": len(dataset_treino),
    "exemplos_validacao": len(dataset_validacao),
    "passos_executados": resultado.global_step,
    "melhor_passo": melhor_passo,
    "melhor_eval_loss": melhor_perda,
    "lote_efetivo": lote_efetivo,
    "learning_rate": argumentos.learning_rate,
    "lora": {"r": lora.r, "alpha": lora.lora_alpha, "dropout": lora.lora_dropout},
    "historico": historico,
}

with open(os.path.join(FINAL_MODEL_DIR, "resumo_treino.json"), "w", encoding="utf-8") as arquivo:
    json.dump(resumo, arquivo, ensure_ascii=False, indent=2)

for nome in sorted(os.listdir(FINAL_MODEL_DIR)):
    caminho = os.path.join(FINAL_MODEL_DIR, nome)
    print(f"{nome:35s} {os.path.getsize(caminho) / 1024**2:8.2f} MB")

print("\nsalvo em:", FINAL_MODEL_DIR)

## 15. Teste rápido

Duas verificações: uma pergunta de conhecimento, que deve vir em português e sem chamar ferramenta,
e uma tarefa agêntica, que deve produzir um bloco `<tool_call>`.

Repare que `enable_thinking` **não** é informado. O padrão faz o prompt de geração terminar em
`<|im_start|>assistant\n`, exatamente a forma usada no treino depois da limpeza da célula 6. Passar
`enable_thinking=False` acrescentaria um `<think></think>` que o modelo não viu nessa posição.

In [ ]:
modelo.config.use_cache = True
modelo.eval()


def responder(mensagens: list[dict], ferramentas: list[dict] | None = None, maximo: int = 320) -> str:
    codificado = tokenizer.apply_chat_template(
        mensagens,
        tools=ferramentas,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    # Conforme a versão do transformers isto vem como tensor puro ou como
    # BatchEncoding; return_dict passou a ser True por padrão na série 5.
    if torch.is_tensor(codificado):
        codificado = {"input_ids": codificado}

    entrada = {chave: valor.to(modelo.device) for chave, valor in codificado.items()}
    tamanho = entrada["input_ids"].shape[-1]

    with torch.no_grad():
        saida = modelo.generate(
            **entrada,
            max_new_tokens=maximo,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(saida[0][tamanho:], skip_special_tokens=False)

PROMPT_ASSISTENTE = next(
    m["content"]
    for r in treino_bruto
    if r["corpus"] == "conhecimento"
    for m in r["messages"]
    if m["role"] == "system"
)

registro_agentico = next(r for r in treino_bruto if r["corpus"] == "agentico")
PROMPT_AGENTE = registro_agentico["messages"][0]["content"]
FERRAMENTAS = registro_agentico["tools"]

print("=" * 70)
print("CONHECIMENTO")
print("=" * 70)
print(
    responder(
        [
            {"role": "system", "content": PROMPT_ASSISTENTE},
            {"role": "user", "content": "Qual a diferença entre SEEK e LOCATE no VFP9?"},
        ]
    )
)

print()
print("=" * 70)
print("AGÊNTICO (deve emitir <tool_call>)")
print("=" * 70)
print(
    responder(
        [
            {"role": "system", "content": PROMPT_AGENTE},
            {"role": "user", "content": "A rotina F_TextoEsquerda não está funcionando. Corrija."},
        ],
        ferramentas=FERRAMENTAS,
    )
)

## 16. Opcional: modelo completo em 16 bits

Só é necessário para exportar depois em GGUF ou para servir sem suporte a LoRA. Consome cerca de
16 GB de RAM e 16 GB no Drive, e a gravação é lenta. Para uso normal, o adaptador da célula 14 basta.

In [ ]:
GERAR_MODELO_COMPLETO = False

if GERAR_MODELO_COMPLETO:
    from peft import PeftModel

    del modelo, treinador
    torch.cuda.empty_cache()

    os.makedirs(MERGED_MODEL_DIR, exist_ok=True)

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="cpu",
        trust_remote_code=True,
        **{chave_dtype: torch.float16},
    )

    combinado = PeftModel.from_pretrained(base, FINAL_MODEL_DIR)
    combinado = combinado.merge_and_unload()

    combinado.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True, max_shard_size="4GB")
    tokenizer.save_pretrained(MERGED_MODEL_DIR)

    print("modelo completo em:", MERGED_MODEL_DIR)
else:
    print("pulado. Defina GERAR_MODELO_COMPLETO = True para gerar.")